In [72]:
# make sure jupyter server is installed in the environment
# then install dependencies
%pip install -r requirements.txt --quiet

# make sure all dependencies are installed
import pandas as pd
import nltk
from nltk import word_tokenize
import sklearn as sk
import numpy as np
import matplotlib.pyplot as plt
from nltk.stem import PorterStemmer, WordNetLemmatizer
from sklearn.model_selection import train_test_split


# download nltk resources
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')

# loading the datasets
negative_df = pd.read_csv('./data/processedNegative.csv', header=None)
positive_df = pd.read_csv('./data/processedPositive.csv', header=None)
neutral_df = pd.read_csv('./data/processedNeutral.csv', header=None)

# transposing the datasets to have tweets in rows
negative_df = negative_df.transpose().rename(columns={0: 'tweet'})
positive_df = positive_df.transpose().rename(columns={0: 'tweet'})
neutral_df = neutral_df.transpose().rename(columns={0: 'tweet'})

# drop any rows with missing values
negative_df = negative_df.dropna().reset_index(drop=True)
positive_df = positive_df.dropna().reset_index(drop=True)
neutral_df = neutral_df.dropna().reset_index(drop=True)

# display the first few rows of each dataset in a dataframe like format
display_df = pd.DataFrame({
    'neutral': neutral_df['tweet'].head(),
    'positive': positive_df['tweet'].head(),
    'negative': negative_df['tweet'].head(),
})
display(display_df)

ERROR: Could not find a version that satisfies the requirement bcc==0.29.1 (from versions: 0.1.7, 0.1.8, 0.1.10)
ERROR: No matching distribution found for bcc==0.29.1
Note: you may need to restart the kernel to use updated packages.


[nltk_data] Downloading package punkt_tab to /home/samy/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /home/samy/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /home/samy/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


,neutral,positive,negative
0,Pak PM survives removal scare,An inspiration in all aspects: Fashion,How unhappy some dogs like it though
1,but court orders further probe into corruptio...,fitness,talking to my over driver about where I'm goin...
2,Supreme Court quashes criminal complaint again...,beauty and personality. :)KISSES TheFashionIcon,Does anybody know if the Rand's likely to fall...
3,Art of Living's fights back over Yamuna floodp...,Apka Apna Awam Ka Channel Frankline Tv Aam Adm...,I miss going to gigs in Liverpool unhappy
4,livid.,Beautiful album from the greatest unsung guit...,There isnt a new Riverdale tonight ? unhappy


### Text Preprocessing

we're going to generate multiple datasets using different preprocessing techniques

##### 1. Text Cleaning

In [73]:
def remove_mentions(tweet):
    """Remove Twitter mentions from a tweet."""
    return ' '.join(word for word in tweet.split() if not word.startswith('@'))

def remove_urls(tweet):
    """Remove URLs from a tweet."""
    return ' '.join(word for word in tweet.split() if not word.startswith('http'))

def remove_hashtags(tweet):
    """Remove hashtags from a tweet."""
    return ' '.join(word for word in tweet.split() if not word.startswith('#'))

def remove_punctuation(tweet):
    """Remove punctuation from a tweet."""
    import string
    return tweet.translate(str.maketrans('', '', string.punctuation))

def convert_to_lowercase(tweet):
    """Convert all characters in a tweet to lowercase."""
    return tweet.lower()

# create a cleaner dataset
def clean_tweet(tweet):
    tweet = str(tweet)
    tweet = remove_mentions(tweet)
    tweet = remove_urls(tweet)
    tweet = remove_hashtags(tweet)
    tweet = remove_punctuation(tweet)
    tweet = convert_to_lowercase(tweet)
    return tweet

In [74]:
# apply cleaning to all datasets
neutral_df['cleaned_tweet'] = neutral_df['tweet'].apply(clean_tweet)
positive_df['cleaned_tweet'] = positive_df['tweet'].apply(clean_tweet)
negative_df['cleaned_tweet'] = negative_df['tweet'].apply(clean_tweet)

# drop null values that may have been introduced during cleaning
neutral_df = neutral_df.dropna().reset_index(drop=True)
positive_df = positive_df.dropna().reset_index(drop=True)
negative_df = negative_df.dropna().reset_index(drop=True)

# display the first few rows of each cleaned dataset in a dataframe like format
cleaned_display_df = pd.DataFrame({
    'neutral': neutral_df['cleaned_tweet'].head(),
    'positive': positive_df['cleaned_tweet'].head(),
    'negative': negative_df['cleaned_tweet'].head(),
})
display(cleaned_display_df)
# display count of tweets in each cleaned dataset
print(f"Neutral tweets: {len(neutral_df)}")
print(f"Positive tweets: {len(positive_df)}")
print(f"Negative tweets: {len(negative_df)}")

,neutral,positive,negative
0,pak pm survives removal scare,an inspiration in all aspects fashion,how unhappy some dogs like it though
1,but court orders further probe into corruption...,fitness,talking to my over driver about where im going...
2,supreme court quashes criminal complaint again...,beauty and personality kisses thefashionicon,does anybody know if the rands likely to fall ...
3,art of livings fights back over yamuna floodpl...,apka apna awam ka channel frankline tv aam adm...,i miss going to gigs in liverpool unhappy
4,livid,beautiful album from the greatest unsung guita...,there isnt a new riverdale tonight unhappy


Neutral tweets: 1569
Positive tweets: 1183
Negative tweets: 1116


#### 2. Tokenizer

###### 2.0 Prepare datasets

In [75]:
neutral_df['tokens'] = neutral_df['cleaned_tweet'].apply(word_tokenize)
positive_df['tokens'] = positive_df['cleaned_tweet'].apply(word_tokenize)
negative_df['tokens'] = negative_df['cleaned_tweet'].apply(word_tokenize)

neutral_df['tokens'].head()

0                  [pak, pm, survives, removal, scare]
1    [but, court, orders, further, probe, into, cor...
2    [supreme, court, quashes, criminal, complaint,...
3    [art, of, livings, fights, back, over, yamuna,...
4                                              [livid]
Name: tokens, dtype: object

###### 2.1 Stemming

In [76]:
stemmer = PorterStemmer()

In [77]:
stemming_neutral = neutral_df.copy()
stemming_positive = positive_df.copy()
stemming_negative = negative_df.copy()
stemming_neutral['stemmed_tokens'] = stemming_neutral['tokens'].apply(lambda tokens: [stemmer.stem(token) for token in tokens])
stemming_positive['stemmed_tokens'] = stemming_positive['tokens'].apply(lambda tokens: [stemmer.stem(token) for token in tokens])
stemming_negative['stemmed_tokens'] = stemming_negative['tokens'].apply(lambda tokens: [stemmer.stem(token) for token in tokens])

stemming_neutral['stemmed_tokens'].head()

0                      [pak, pm, surviv, remov, scare]
1    [but, court, order, further, probe, into, corr...
2    [suprem, court, quash, crimin, complaint, agai...
3    [art, of, live, fight, back, over, yamuna, flo...
4                                              [livid]
Name: stemmed_tokens, dtype: object

###### 2.2 Lemmatization

In [78]:
lemmatizer = WordNetLemmatizer()

lemmatization_neutral = neutral_df.copy()
lemmatization_positive = positive_df.copy()
lemmatization_negative = negative_df.copy()
lemmatization_neutral['lemmatized_tokens'] = lemmatization_neutral['tokens'].apply(lambda tokens: [lemmatizer.lemmatize(token) for token in tokens])
lemmatization_positive['lemmatized_tokens'] = lemmatization_positive['tokens'].apply(lambda tokens: [lemmatizer.lemmatize(token) for token in tokens])
lemmatization_negative['lemmatized_tokens'] = lemmatization_negative['tokens'].apply(lambda tokens: [lemmatizer.lemmatize(token) for token in tokens])

lemmatization_neutral['lemmatized_tokens'].head()

0                  [pak, pm, survives, removal, scare]
1    [but, court, order, further, probe, into, corr...
2    [supreme, court, quashes, criminal, complaint,...
3    [art, of, living, fight, back, over, yamuna, f...
4                                              [livid]
Name: lemmatized_tokens, dtype: object

### Cosine Similarity

### Machine Learning

In [79]:
# split datasets into training and testing sets (80% train, 20% test)
neutral_train, neutral_test = train_test_split(neutral_df, test_size=0.2, random_state=42)
positive_train, positive_test = train_test_split(positive_df, test_size=0.2, random_state=42)
negative_train, negative_test = train_test_split(negative_df, test_size=0.2, random_state=42)